# 1.2 – MNIST Classifier Evaluation

Loads the pre-trained `MNISTClassifier` from its Lightning checkpoint, evaluates it on the MNIST test set, and visualises the results.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

from models.classifiers import MNISTClassifier
from training.lit_classifier import LitClassifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT = "../checkpoints/classifier/MNIST-Classifier-032-0.0144.ckpt"
DATA_DIR = "../data"
print(f"Using device: {DEVICE}")

## Load model from Lightning checkpoint

In [ ]:
lit = LitClassifier.load_from_checkpoint(CHECKPOINT, model=MNISTClassifier())
classifier = lit.model.to(DEVICE).eval()
n_params = sum(p.numel() for p in classifier.parameters())
print(f"MNISTClassifier loaded — {n_params:,} parameters")

## Evaluate on test set

In [ ]:
test_ds = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transforms.ToTensor())
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=4)

all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        logits = classifier(x.to(DEVICE))
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(y)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
accuracy = (all_preds == all_labels).mean()
print(f"Test accuracy: {accuracy:.4f}  ({accuracy*100:.2f}%)")

## Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(
    ax=ax, colorbar=False, cmap="Blues"
)
ax.set_title(f"MNIST Classifier — Test Accuracy: {accuracy*100:.2f}%")
plt.tight_layout()
plt.show()

## Sample predictions

In [ ]:
# Pick 20 random test images and show their true vs. predicted label
rng = np.random.default_rng(0)
indices = rng.choice(len(test_ds), size=20, replace=False)

images = torch.stack([test_ds[i][0] for i in indices]).to(DEVICE)
with torch.no_grad():
    preds = classifier(images).argmax(1).cpu().numpy()
true = np.array([test_ds[i][1] for i in indices])

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for ax, img, t, p in zip(axes.flat, images.cpu(), true, preds):
    ax.imshow(img.squeeze(), cmap="gray")
    color = "green" if t == p else "red"
    ax.set_title(f"{p}", color=color, fontsize=9)
    ax.axis("off")
fig.suptitle("Predicted labels (green = correct, red = wrong)", fontsize=11)
plt.tight_layout()
plt.show()